# 1.1. Señales continuas y discretas

> La información de una señal está contenida en un patrón de variaciones con una forma determinada.


## Introducción


**Ejemplo 1**: señal de voz humana (variaciones de presión acústica).


Distintos patrones de variación producen distintos sonidos:



In [54]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))

from utils.plot_helpers import style_math_axes

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, Span, HoverTool, Title
import numpy as np

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True);

# ==== PARÁMETROS ====
sr = 16000  # frecuencia de muestreo
dur = 0.5   # segundos por letra
t_segment = np.linspace(0, dur, int(sr*dur), endpoint=False)

def envelope(t, attack=0.01, decay=0.25):
    env = np.ones_like(t)
    env *= np.minimum(1.0, t/attack)
    env *= np.exp(-t/decay)
    return env

# ==== SINTETIZAR LETRAS ====
# L
f0 = 120.0
harmonics = [1.0, 0.6, 0.3, 0.15]
sig_L = np.zeros_like(t_segment)
for i, a in enumerate(harmonics, start=1):
    sig_L += a * np.sin(2*np.pi*f0*i*t_segment)
sig_L *= envelope(t_segment, attack=0.02, decay=0.4)

# T
noise = np.random.normal(0, 1, size=t_segment.shape)
burst = np.zeros_like(t_segment)
burst_len = int(0.02*sr)
burst[:burst_len] = noise[:burst_len] * np.hanning(burst_len)
sig_T = burst * envelope(t_segment, attack=0.001, decay=0.1) + 0.02 * np.sin(2*np.pi*800*t_segment) * np.exp(-5*t_segment)

# I
f0_i = 160.0
formants = [2200, 3000]
sig_I = 0.6*np.sin(2*np.pi*f0_i*t_segment) * envelope(t_segment, attack=0.01, decay=0.5)
for fm in formants:
    sig_I += 0.3 * np.sin(2*np.pi*fm*t_segment) * envelope(t_segment, attack=0.01, decay=0.4)

# Normalizar
def norm(x): return x / (np.max(np.abs(x)) + 1e-9)
sig_L, sig_T, sig_I = map(norm, [sig_L, sig_T, sig_I])

# ==== CONCATENAR ====
gap = np.zeros(int(0.05*sr))
wave = np.concatenate([sig_L, gap, sig_T, gap, sig_I])
t_full = np.linspace(0, len(wave)/sr, len(wave), endpoint=False)

# ==== LÍMITES ====
start_L = 0
end_L = len(sig_L)
start_T = end_L + len(gap)
end_T = start_T + len(sig_T)
start_I = end_T + len(gap)
end_I = start_I + len(sig_I)

# ==== FUENTES DE DATOS ====
src_L = ColumnDataSource(data=dict(time=t_full[start_L:end_L], amp=sig_L, letter=["L"]*len(sig_L)))
src_T = ColumnDataSource(data=dict(time=t_full[start_T:end_T], amp=sig_T, letter=["T"]*len(sig_T)))
src_I = ColumnDataSource(data=dict(time=t_full[start_I:end_I], amp=sig_I, letter=["I"]*len(sig_I)))

# ==== GRAFICAR ====
TOOLS = "pan,wheel_zoom,box_zoom,reset,save,hover"
p = figure(title="Onda de audio: Pronunciación sintetizada de las letras L, T e I", 
            sizing_mode="stretch_width",
            max_width=600,
            height=400, 
            tools=TOOLS)

# Líneas por letra
p.line('time', 'amp', source=src_L, color="blue", line_width=2)
p.line('time', 'amp', source=src_T, color="green", line_width=2)
p.line('time', 'amp', source=src_I, color="red", line_width=2)

# Líneas divisorias
v1 = Span(location=end_L/sr, dimension='height', line_dash='dashed', line_width=1, line_color="gray")
v2 = Span(location=end_T/sr, dimension='height', line_dash='dashed', line_width=1, line_color="gray")
p.add_layout(v1)
p.add_layout(v2)

# Etiquetas ejes
p.xaxis.axis_label = "Tiempo (s)"
p.yaxis.axis_label = "Amplitud"
p.toolbar.logo = None

style_math_axes(
    p,
    x_range=(0, len(wave)/sr),
    y_range=(-1, 1),
    xlabel="Tiempo (s)",
    ylabel="Amplitud"
)

# ==== HOVER TOOL ====
hover = HoverTool(
    tooltips=[
        ("Letra", "@letter"),
        ("Tiempo (s)", "@time{0.000}"),
        ("Amplitud", "@amp{0.000}")
    ],
    mode="vline"  # muestra valores verticalmente alineados
)
p.add_tools(hover)

# ==== CAPTION (debajo) ====
# p.add_layout(Title(text="Distintos patrones de variación producen distintos sonidos.", 
#                    text_font_size="10pt", text_color="gray"), 'below')

# ==== MOSTRAR ====
show(p)


**Ejemplo 2**: fotografía (blanco y negro).

Señal bidimensional. Variaciones del nivel de gris.

In [56]:
from PIL import Image
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, LinearColorMapper, ColorBar

output_notebook(verbose=False, hide_banner=True);

# ===== LOAD IMAGE =====
img = Image.open("figures\\cameraman.png").convert("L")
img_array = np.array(img)
img_array = np.flipud(img_array)  # flip vertically

h, w = img_array.shape

# ===== NORMALIZE FOR COLOR MAPPING =====
color_mapper = LinearColorMapper(palette="Greys256", low=0, high=255);

# ===== CREATE FIGURE =====
p = figure(
    width=400, height=400,
    x_range=(0, w), y_range=(0, h),
    tools="hover,pan,box_zoom,reset,save"
);

# Remove ticks and grid
p.axis.visible = False
p.grid.visible = False

# Display image
p.image(image=[img_array], x=0, y=0, dw=w, dh=h, color_mapper=color_mapper);

# Add color bar
color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12, location=(0,0));
# p.add_layout(color_bar, 'right')

# Hover tooltips (pixel coordinates)
hover = p.select_one(HoverTool);
hover.tooltips = [
    ("Intensity", "@image")  # @image references the value under the mouse
]
hover.mode = 'mouse';

show(p);


In [57]:
from PIL import Image
import numpy as np
import plotly.graph_objects as go

# ===== LOAD IMAGE =====
img = Image.open("figures\\cameraman.png").convert("L")
img_array = np.array(img)

# Flip vertically so top is at top
img_array = np.flipud(img_array)

h, w = img_array.shape
X, Y = np.meshgrid(np.arange(w), np.arange(h))
Z = img_array  # intensity as height

# ===== PLOTLY SURFACE =====
fig = go.Figure(data=[
    go.Surface(
        x=X,
        y=Y,
        z=Z,
        colorscale='gray',
        showscale=False,
        colorbar=dict(title='Intensity')
    )
])

fig.update_layout(
    # title='Variaciones del nivel de gris de la imagen',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Intensity',

        # <<--- camera / default view
        camera=dict(eye=dict(x=.0, y=-.9, z=2.2))
    ),
    width=600,
    height=400,
    margin=dict(l=0, r=0, b=0, t=0)
)

fig.show()


---

**Representación matemática**: funciones de una o más variables independientes.

>Aquí sólo nos ocuparemos de señales con una variable independiente.

En general, consideramos que es el tiempo: $x(t)$ (aunque no tiene por qué).

Vamos a considerar **dos tipos básicos de señales**:
  
- **Continuas**: la variable independiente es continua $\Rightarrow$ se definen para una sucesión continua de valores de la variable independiente.

  **Notación**: $\boxed{x(t)}$, con $t\in\mathbb{R}$.
    
  ```{figure} figures/T1/2_1_fig4  
  ---
    width: 60%
  ---

  ```

    Ejemplos: señal de voz con el tiempo, presión atmosférica con altitud.



- **Discretas**: la variable independiente sólo toma un conjunto discreto de valores. También se las llama **secuencia discreta**.

  **Notación**: $\boxed{x[n]}$, con $n\in\Z$.

    
  ```{figure} figures/T1/2_1_fig5 
  ---
    width: 60%
  ---

  ```

    Ejemplos: valor del IBEX-35 al final de cada sesión, muestreo de señales continuas (muestras equiespaciadas).

  

Se mostrarán las señales continuas y discretas de forma paralela, para irlas relacionando. En el capítulo de muestreo (7), veremos cómo se puede pasar de unas a otras, idealmente sin error.

(clases_senales)=
## Clases de señales

Veremos a continuación distintos tipos de señales que tendrán importancia a lo largo de toda la asignatura.

  :::{important .simple icon=false}  **Real e imaginaria**: <span style="font-weight: normal;">simetría respecto a la conjugación.</span>
  
    - **Real pura**: simétrica respecto a la conjugación.

      \begin{equation}
	 x^*(t)=x(t),
      \end{equation}
      \begin{equation}
	 x^*[n]=x[n].
      \end{equation}

    - **Imaginaria pura**: antisimétrica respecto a la conjugación.

      \begin{equation}
	 x^*(t)=-x(t),
      \end{equation}
      \begin{equation}
	 x^*[n]=-x[n].
      \end{equation}

  :::

  :::{important .simple icon=false}  **Par e impar**: <span style="font-weight: normal;">simetría respecto a la inversión en el tiempo. Se aplica a señales reales.</span>
   
    - **Par**: simétrica respecto al eje de ordenadas.

      \begin{equation}
	 x(-t)=x(t),
      \end{equation}
      \begin{equation}
	 x[-n]=x[n].
      \end{equation}
    
    
      ```{figure} figures/T1/2_1_fig6 
      ---
      width: 60%
      ---

      ```

    
    - **Impar**: antisimétrica respecto al eje de ordenadas.

      \begin{equation}
    x(-t)=-x(t),
      \end{equation}
      \begin{equation}
    x[-n]=-x[n].
      \end{equation}
      
    
      ```{figure} figures/T1/2_1_fig7 
      ---
      width: 60%
      ---

      ```

          
      En el origen, como $x(0)=-x(0)$, se cumple:

        \begin{equation}
      x(0)=0,
        \end{equation}
        \begin{equation}
      x[0]=0,
        \end{equation}
  :::
  
:::{important .simple icon=false}  **Hermítica y antihermítica**: <span style="font-weight: normal;"> Equivalente para señales complejas.</span>

  - **Hermítica**: simétrica respecto al eje de ordenadas y la conjugación.

      \begin{equation}
  x^*(-t)=x(t),
      \end{equation}
      \begin{equation}
  x^*[-n]=x[n].
      \end{equation}
    
  - **Antihermítica**: antisimétrica respecto al eje de ordenadas y la conjugación.

    \begin{equation}
x^*(-t)=-x(t),
    \end{equation}
    \begin{equation}
x^*[-n]=-x[n].
    \end{equation}

:::
  

Toda señal se puede poner como suma de sus partes real e imaginaria, par e impar, hermítica y antihermítica:

  \begin{equation}
    x(t)=x_r(t)+x_i(t)=\Re\{x(t)\}+\Im\{x(t)\}.
  \end{equation}
  \begin{equation}
    x(t)=x_e(t)+x_o(t)=\mathcal{Ev}\{x(t)\}+\mathcal{Od}\{x(t)\},\quad x(t)\in\mathbb{R}.
  \end{equation}
  \begin{equation}
    x(t)=x_h(t)+x_a(t).
  \end{equation}
Las expresiones para el caso discreto son equivalentes.

Cálculo de la parte par e impar de una señal:

```{math}
\begin{cases}x(t)=x_e(t)+x_o(t),\\x(-t)=x_e(-t)+x_o(-t)=x_e(t)-x_o(t)\end{cases}
```

```{math}
\begin{equation*}
 \Downarrow 
\end{equation*} 
```


  Parte par e impar de una señal:
```{math}
\begin{cases}x_e(t)=\frac{1}{2}\left[x(t)+x(-t)\right],\\x_o(t)=\frac{1}{2}\left[x(t)-x(-t)\right]\end{cases}
```

Se cumple:
```{math}
x_e(0)=x(0),\qquad x_e[0]=x[0].
```
```{math}
x_o(0)=0,\qquad x_o[0]=0.
```

Parte real e imaginaria:
```{math}
\begin{cases}x_r(t)=\frac{1}{2}\left[x(t)+x^*(t)\right],\\x_i(t)=\frac{1}{2}\left[x(t)-x^*(t)\right]\end{cases}
```

Parte hermítica y antihermítica:
```{math}
\begin{cases} x_h(t)=\frac{1}{2}\left[x(t)+x^*(-t)\right],\\x_a(t)=\frac{1}{2}\left[x(t)-x^*(-t)\right]\end{cases}
```


Para el caso discreto las expresiones son equivalentes.

**Ejemplo**: Cálculo de la parte par e impar de una señal:

$x[n]=\begin{cases} 1, & n\ge 0,\\0, & n<0\end{cases}$

 ```{figure} figures/T1/2_1_fig8 
 ---
 width: 60%
 ---

 ```

$x[-n]=\begin{cases} 1, & n\le 0,\\0, & n>0\end{cases}$

 ```{figure} figures/T1/2_1_fig9 
 ---
 width: 60%
 ---

 ```

$x_e[n]=\begin{cases} 1/2, & n<0,\\1, & n=0,\\1/2, & n>0\end{cases}$

 ```{figure} figures/T1/2_1_fig10 
 ---
 width: 60%
 ---

 ```

$x_o[n]=\begin{cases} -1/2, & n<0,\\0, & n=0,\\1/2, & n>0\end{cases}$

 ```{figure} figures/T1/2_1_fig11 
 ---
 width: 60%
 ---
 ```


In [58]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))

from utils.plot_helpers import style_math_axes

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Select, Div
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. EJE TEMPORAL Y SEÑALES BASE
# ==============================================================================

t_min, t_max = -4.0, 4.0
num_points = 2000
t = np.linspace(t_min, t_max, num_points)

signals = {}

# Señal real par: x(t) = x(-t)
x = np.exp(-0.25*t**2) * np.cos(2*np.pi*0.7*t)
signals["Real par"] = dict(real=x, imag=np.zeros_like(t))

# Señal real impar: x(t) = -x(-t)
x = np.exp(-0.25*t**2) * np.sin(2*np.pi*0.7*t)
signals["Real impar"] = dict(real=x, imag=np.zeros_like(t))

# Señal puramente imaginaria
x = 1j * np.where(t>0, np.exp(-0.25*t**2), 2*np.exp(-0.5*t**2)) * np.sin(2*np.pi*0.7*t)
signals["Imaginaria pura"] = dict(real=np.zeros_like(t), imag=np.imag(x))

# Señal hermítica: x(t) = x*(-t)
# Parte real par + parte imaginaria impar
real = np.exp(-0.25*t**2) * np.cos(2*np.pi*0.6*t)
imag = np.exp(-0.25*t**2) * np.sin(2*np.pi*0.6*t)
signals["Hermítica"] = dict(real=real, imag=imag)

# Señal antihermítica: x(t) = -x*(-t)
# Parte real impar + parte imaginaria par
real = np.exp(-0.25*t**2) * np.sin(2*np.pi*0.6*t)
imag = np.exp(-0.25*t**2) * np.cos(2*np.pi*0.6*t)
signals["Antihermítica"] = dict(real=real, imag=imag)

# Señal compleja general
real = np.exp(-0.15*(t+1)**2) * np.cos(2*np.pi*0.45*t)
imag = 0.7*np.exp(-0.15*(t-1)**2) * np.sin(2*np.pi*0.8*t)
signals["Compleja general"] = dict(real=real, imag=imag)

initial = "Real par"

source = ColumnDataSource(data=dict(
    t=t,
    real=signals[initial]["real"],
    imag=signals[initial]["imag"],
    real_mirror=signals[initial]["real"][::-1],
    imag_mirror=signals[initial]["imag"][::-1],
))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=600,
    title="Simetrías de señales reales y complejas",
    tools="pan,wheel_zoom,reset"
)

p.line("t", "real", source=source, line_width=3, color="blue",
       legend_label="Re{x(t)}")

p.line("t", "imag", source=source, line_width=3, color="red",
       legend_label="Im{x(t)}")

p.line("t", "real_mirror", source=source, line_width=2, color="blue",
       line_dash="dashed", alpha=0.45, legend_label="Re{x(-t)}")

p.line("t", "imag_mirror", source=source, line_width=2, color="red",
       line_dash="dashed", alpha=0.45, legend_label="Im{x(-t)}")

p.line(t, np.zeros_like(t), color="black", alpha=0.4)

p.xaxis.axis_label = "t"
p.yaxis.axis_label = "amplitud"
p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"

style_math_axes(p, x_range=(t_min, t_max), y_range=(-1.5, 1.5), xlabel="t", ylabel='amplitud')
# add_math_ticks(p, yticks=[-1, 1], ytick_labels=["-1", "1"], tick_len=5)

# ==============================================================================
# 3. CONTROLES Y TEXTO
# ==============================================================================

select = Select(
    title="Tipo de señal",
    value=initial,
    options=list(signals.keys()),
    width=260
)

info = Div(width=600)

texts = {
    "Real par": """
    <b>Señal real par</b><br>
    La señal cumple: <br>
    <span style="font-size:18px;">x(t) = x(-t)</span><br><br>
    La señal es simétrica respecto al eje vertical.
    """,

    "Real impar": """
    <b>Señal real impar</b><br>
    La señal cumple: <br>
    <span style="font-size:18px;">x(t) = -x(-t)</span><br><br>
    La señal cambia de signo al reflejarla respecto a t = 0.
    """,

    "Imaginaria pura": """
    <b>Señal imaginaria pura</b><br>
    La parte real es cero: <br>
    <span style="font-size:18px;">Re{x(t)} = 0</span><br><br>
    Toda la información está en la parte imaginaria.
    """,

    "Hermítica": """
    <b>Señal hermítica</b><br>
    La señal cumple: <br>
    <span style="font-size:18px;"> \\( x(t)=x^*(-t) \\) </span><br><br>
    Esto implica que la parte real es par y la parte imaginaria es impar.
    """,

    "Antihermítica": """
    <b>Señal antihermítica</b><br>
    La señal cumple: <br>
    <span style="font-size:18px;">
    \\( x(t) = -x^*(-t) \\)</span><br><br>
    Esto implica que la parte real es impar y la parte imaginaria es par.
    """,

    "Compleja general": """
    <b>Señal compleja general</b><br>
    No cumple necesariamente ninguna simetría especial.<br><br>
    Tiene parte real e imaginaria independientes.
    """
}

info.text = texts[initial]

callback = CustomJS(
    args=dict(
        source=source,
        select=select,
        signals=signals,
        texts=texts,
        info=info
    ),
    code="""
    const name = select.value;
    const sig = signals[name];

    source.data["real"] = sig["real"];
    source.data["imag"] = sig["imag"];

    source.data["real_mirror"] = [...sig["real"]].reverse();
    source.data["imag_mirror"] = [...sig["imag"]].reverse();

    info.text = texts[name];

    source.change.emit();
    """
)

select.js_on_change("value", callback)

layout = column(
    p, select,
    info
)

show(layout)




A continuación veremos otros tipos de señales: periódicas, de energía y de potencia.

(periodicas)=
## Señales periódicas


Dada su importancia las vemos aparte.

:::{important .simple icon=false}  **Definición de Señal periódica continua**
  Una señal **continua**, $x(t)$, es **periódica** si:
  \begin{equation}\label{periodica_continua}
    \exists\ T\in\mathbb{R}^+ /\quad \boxed{x(t)=x(t+T)},\quad \forall t\in\mathbb{R}.
  \end{equation}
  $T$: **período** de la señal.
:::

  **Ejemplos**:
  
  ```{figure} figures/T1/2_1_fig12 
  ---
  width: 60%
  ---

  ```

  

  Si $x(t)$ es periódica con período $T$, también lo es con período $mT,\ m\in\N$.

  $T_0$: **período fundamental** de la señal. Valor más pequeño de $T$ para el que se satisface:
  \begin{equation}
    x(t)=x(t+T_0).
  \end{equation}

:::{note} Nota
Si $x(t)$ es constante, no está definido $T_0$, ya que es periódica para cualquier $T$.
:::
  
:::{important .simple icon=false} **Definición de Señal aperiódica continua**
Una señal continua que no es periódica.
:::

**Ejemplos:**
```{math}
x(t)=\begin{cases} \cos(t), & t<0,\\ \sin(t), & t\ge 0\end{cases}
```

Se cumple que $\cos(t)=\cos(t+2\pi)$ para $t<-2\pi$ y $\sin(t)=\sin(t+2\pi)$, para $t\ge0$, pero no se cumple $x(t)=x(t+2\pi),\ \forall t$.


```{figure} figures/T1/2_1_fig13 
---
width: 60%
---

```


In [60]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Select, Div, Slider
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJE TEMPORAL
# ==============================================================================

t_min, t_max = -4.0, 4.0
num_points = 2000
t = np.linspace(t_min, t_max, num_points)

initial = "Coseno"

A0 = 1.0
T0 = 1.0
tau0 = 0.0


def continuous_signal(name, t_values, A, T):
    """Genera las señales iniciales en Python."""

    if name == "Coseno":
        return A * np.cos(2 * np.pi * t_values / T)

    if name == "Seno":
        return A * np.sin(2 * np.pi * t_values / T)

    if name == "Suma de armónicos":
        return A * (
            np.cos(2 * np.pi * t_values / T)
            + 0.4 * np.cos(4 * np.pi * t_values / T)
        )

    if name == "Cuadrada":
        return A * np.where(
            np.sin(2 * np.pi * t_values / T) >= 0,
            1,
            -1
        )

    if name == "No periódica":
        return (
            A
            * np.exp(-0.4 * t_values**2)
            * np.cos(2 * np.pi * t_values / T)
        )

    raise ValueError(f"Tipo de señal desconocido: {name}")


x0 = continuous_signal(initial, t, A0, T0)
x_shift0 = continuous_signal(initial, t + tau0, A0, T0)

source = ColumnDataSource(
    data=dict(
        t=t,
        x=x0,
        x_shift=x_shift0
    )
)

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=420,
    width=700,
    title="Periodicidad continua: comparación entre x(t) y x(t+τ)",
    tools="pan,wheel_zoom,box_zoom,reset"
)

p.line(
    "t",
    "x",
    source=source,
    line_width=3,
    color="blue",
    legend_label="x(t)"
)

p.line(
    "t",
    "x_shift",
    source=source,
    line_width=2,
    color="red",
    line_dash="dashed",
    alpha=0.7,
    legend_label="x(t+τ)"
)

p.line(
    t,
    np.zeros_like(t),
    color="black",
    alpha=0.4
)

p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"

style_math_axes(
    p,
    x_range=(t_min, t_max),
    y_range=(-2.5, 2.5),
    xlabel="t",
    ylabel="amplitud"
)

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

select = Select(
    title="Tipo de señal",
    value=initial,
    options=[
        "Coseno",
        "Seno",
        "Suma de armónicos",
        "Cuadrada",
        "No periódica"
    ],
    width=210
)

amp_slider = Slider(
    title="Amplitud A",
    start=0.2,
    end=2.0,
    value=A0,
    step=0.1,
    width=190
)

period_slider = Slider(
    title="Período T",
    start=0.5,
    end=3.0,
    value=T0,
    step=0.1,
    width=190
)

shift_slider = Slider(
    title="Desplazamiento τ",
    start=-3.0,
    end=3.0,
    value=tau0,
    step=0.1,
    width=210
)

info = Div(width=700)

info.text = """
<div style="
    padding: 10px;
    border-left: 5px solid #2e8b57;
    background-color: #eef8f1;
">
    <b>Las señales coinciden.</b><br>
    τ = 0 es un múltiplo entero de T = 1.<br><br>
    <span style="font-size:18px;">
        x(t) = x(t+τ)
    </span>
</div>
"""

# ==============================================================================
# 4. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        select=select,
        amp_slider=amp_slider,
        period_slider=period_slider,
        shift_slider=shift_slider,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];

    const name = select.value;
    const A = amp_slider.value;
    const T = period_slider.value;
    const tau = shift_slider.value;

    const x = [];
    const xShift = [];

    function signalValue(time) {
        if (name === "Coseno") {
            return A * Math.cos(2 * Math.PI * time / T);
        }

        if (name === "Seno") {
            return A * Math.sin(2 * Math.PI * time / T);
        }

        if (name === "Suma de armónicos") {
            return A * (
                Math.cos(2 * Math.PI * time / T)
                + 0.4 * Math.cos(4 * Math.PI * time / T)
            );
        }

        if (name === "Cuadrada") {
            const s = Math.sin(2 * Math.PI * time / T);
            return A * (s >= 0 ? 1 : -1);
        }

        if (name === "No periódica") {
            return A
                * Math.exp(-0.4 * time * time)
                * Math.cos(2 * Math.PI * time / T);
        }

        return 0;
    }

    let maxDifference = 0;

    for (let i = 0; i < t.length; i++) {
        const ti = t[i];

        const value = signalValue(ti);
        const shiftedValue = signalValue(ti + tau);

        x.push(value);
        xShift.push(shiftedValue);

        const difference = Math.abs(value - shiftedValue);

        if (difference > maxDifference) {
            maxDifference = difference;
        }
    }

    data["x"] = x;
    data["x_shift"] = xShift;

    const isPeriodicSignal = name !== "No periódica";

    /*
    Se comprueba si tau/T está suficientemente cerca de un entero.
    La tolerancia evita problemas debidos a la representación decimal
    de valores como 0.1, 0.2, etc.
    */
    const ratio = tau / T;
    const nearestInteger = Math.round(ratio);
    const isMultiple = Math.abs(ratio - nearestInteger) < 1e-8;

    const curvesCoincide = maxDifference < 1e-6;

    const tauText = tau.toFixed(1);
    const TText = T.toFixed(1);

    if (isPeriodicSignal && isMultiple && curvesCoincide) {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #2e8b57;
                background-color: #eef8f1;
            ">
                <b>Las señales coinciden.</b><br>
                τ = ${tauText} es un múltiplo entero de
                T = ${TText}.<br><br>

                <span style="font-size:18px;">
                    x(t) = x(t+τ)
                </span><br><br>

                En este caso:
                <span style="font-size:17px;">
                    τ = ${nearestInteger}T
                </span>
            </div>
        `;
    }

    else if (isPeriodicSignal) {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #c44;
                background-color: #fff1f1;
            ">
                <b>Las señales no coinciden.</b><br>
                τ = ${tauText} no es un múltiplo entero de
                T = ${TText}.<br><br>

                Para una señal periódica:
                <br>

                <span style="font-size:18px;">
                    x(t) = x(t+τ)
                </span>

                cuando

                <span style="font-size:18px;">
                    τ = mT, &nbsp; m ∈ ℤ.
                </span>
            </div>
        `;
    }

    else if (Math.abs(tau) < 1e-8) {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #2e8b57;
                background-color: #eef8f1;
            ">
                <b>Las señales coinciden porque τ = 0.</b><br><br>

                Toda señal coincide consigo misma cuando no se aplica
                ningún desplazamiento:

                <br><br>

                <span style="font-size:18px;">
                    x(t) = x(t+0)
                </span>

                <br><br>

                Esto no implica que la señal sea periódica.
            </div>
        `;
    }

    else {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #c44;
                background-color: #fff1f1;
            ">
                <b>Señal no periódica.</b><br>
                Para τ = ${tauText}, la señal desplazada no coincide
                con la señal original.<br><br>

                La envolvente exponencial impide que la señal se repita:

                <br><br>

                <span style="font-size:18px;">
                    x(t) ≠ x(t+τ)
                </span>
            </div>
        `;
    }

    source.change.emit();
    """
)

select.js_on_change("value", callback)
amp_slider.js_on_change("value", callback)
period_slider.js_on_change("value", callback)
shift_slider.js_on_change("value", callback)

# ==============================================================================
# 5. DISTRIBUCIÓN
# ==============================================================================

layout = column(
    p,
    row(select, amp_slider),
    row(period_slider, shift_slider),
    info
)

show(layout)



:::{warning .simple icon=false} Ejercicio 1
Estudiar el caso:
```{math}
x(t)=\begin{cases} \cos(t), & t<0,\\ \cos(-t), & t\ge 0 \end{cases}
```
:::

:::{important .simple icon=false} **Definición de Señal periódica discreta**
Una señal **discreta**, $x[n]$, es **periódica** si:
\begin{equation}
  \exists\ N\in\N /\quad \boxed{x[n]=x[n+N]},\quad \forall n\in\Z.
\end{equation}
$N$: **período** de la señal.
:::

**Ejemplos**:

```{figure} figures/T1/2_1_fig14 
---
width: 60%
---

```

Si $x[n]$ es periódica con período $N$, también lo es con período $mN,\ m\in\N$.

$N_0$: **período fundamental** de la secuencia discreta. Valor más pequeño de $N$ para el que se satisface:
\begin{equation}
  x[n]=x[n+N_0].
\end{equation}

Si $x[n]$ es constante, $N_0=1$, que es el período mínimo que puede tener una señal discreta.

:::{important .simple icon=false} **Definición de Señal aperiódica discreta**
Una señal discreta que no es periódica.
:::


In [61]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Select, Div, Slider
from bokeh.layouts import column, row

from utils.plot_helpers import style_math_axes

output_notebook(verbose=False, hide_banner=True)

# ==============================================================================
# 1. EJE DISCRETO
# ==============================================================================

n_min, n_max = -20, 20
n = np.arange(n_min, n_max + 1)

initial = "Coseno"
A0 = 1.0
N0 = 8
k0 = 0


def discrete_signal(name, n_values, A, N):
    """Genera las señales iniciales en Python."""

    if name == "Coseno":
        return A * np.cos(2 * np.pi * n_values / N)

    if name == "Seno":
        return A * np.sin(2 * np.pi * n_values / N)

    if name == "Suma de armónicos":
        return A * (
            np.cos(2 * np.pi * n_values / N)
            + 0.4 * np.cos(4 * np.pi * n_values / N)
        )

    if name == "Cuadrada":
        return A * np.where((n_values % N) < N/2, 1, -1)

    if name == "No periódica":
        return A * np.exp(-0.03 * n_values**2) * np.cos(
            2 * np.pi * n_values / N
        )

    raise ValueError(f"Tipo de señal desconocido: {name}")


x0 = discrete_signal(initial, n, A0, N0)
x_shift0 = discrete_signal(initial, n + k0, A0, N0)

source = ColumnDataSource(
    data=dict(
        n=n,
        x=x0,
        x_shift=x_shift0,
        zeros=np.zeros_like(n, dtype=float)
    )
)

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=430,
    width=700,
    title="Periodicidad discreta: comparación entre x[n] y x[n+k]",
    tools="pan,wheel_zoom,box_zoom,reset"
)

# Líneas auxiliares tipo stem
p.segment(
    x0="n",
    y0="zeros",
    x1="n",
    y1="x",
    source=source,
    line_width=2,
    color="blue",
    alpha=0.65
)

p.scatter(
    "n",
    "x",
    source=source,
    size=8,
    color="blue",
    legend_label="x[n]"
)

p.segment(
    x0="n",
    y0="zeros",
    x1="n",
    y1="x_shift",
    source=source,
    line_width=2,
    color="red",
    line_dash="dashed",
    alpha=0.5
)

p.scatter(
    "n",
    "x_shift",
    source=source,
    size=7,
    color="red",
    marker="diamond",
    alpha=0.75,
    legend_label="x[n+k]"
)

p.line(
    n,
    np.zeros_like(n),
    color="black",
    alpha=0.4
)

p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"

style_math_axes(
    p,
    x_range=(n_min - 1, n_max + 1),
    y_range=(-2.5, 2.5),
    xlabel="n",
    ylabel="amplitud"
)

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

select = Select(
    title="Tipo de señal",
    value=initial,
    options=[
        "Coseno",
        "Seno",
        "Suma de armónicos",
        "Cuadrada",
        "No periódica"
    ],
    width=210
)

amp_slider = Slider(
    title="Amplitud A",
    start=0.2,
    end=2.0,
    value=A0,
    step=0.1,
    width=190
)

period_slider = Slider(
    title="Período N",
    start=2,
    end=16,
    value=N0,
    step=1,
    width=190
)

shift_slider = Slider(
    title="Desplazamiento k",
    start=-16,
    end=16,
    value=k0,
    step=1,
    width=210
)

info = Div(width=700)

# ==============================================================================
# 4. CALLBACK
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        select=select,
        amp_slider=amp_slider,
        period_slider=period_slider,
        shift_slider=shift_slider,
        info=info
    ),
    code="""
    const data = source.data;
    const n = data["n"];

    const name = select.value;
    const A = amp_slider.value;
    const N = period_slider.value;
    const k = shift_slider.value;

    const x = [];
    const xShift = [];

    function signalValue(index) {
        if (name === "Coseno") {
            return A * Math.cos(2 * Math.PI * index / N);
        }

        if (name === "Seno") {
            return A * Math.sin(2 * Math.PI * index / N);
        }

        if (name === "Suma de armónicos") {
            return A * (
                Math.cos(2 * Math.PI * index / N)
                + 0.4 * Math.cos(4 * Math.PI * index / N)
            );
        }

        if (name === "Cuadrada") {
            const k = ((index % N) + N) % N;   // handles negative indices
            return A * (k < N/2 ? 1 : -1);
        }

        if (name === "No periódica") {
            return A
                * Math.exp(-0.03 * index * index)
                * Math.cos(2 * Math.PI * index / N);
        }

        return 0;
    }

    let maxDifference = 0;

    for (let i = 0; i < n.length; i++) {
        const ni = n[i];

        const value = signalValue(ni);
        const shiftedValue = signalValue(ni + k);

        x.push(value);
        xShift.push(shiftedValue);

        const difference = Math.abs(value - shiftedValue);

        if (difference > maxDifference) {
            maxDifference = difference;
        }
    }

    data["x"] = x;
    data["x_shift"] = xShift;

    const isPeriodicSignal = name !== "No periódica";
    const isMultiple = k % N === 0;
    const curvesCoincide = maxDifference < 1e-8;

    if (isPeriodicSignal && isMultiple && curvesCoincide) {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #2e8b57;
                background-color: #eef8f1;
            ">
                <b>Las señales coinciden.</b><br>
                k = ${k} es múltiplo de N = ${N}.<br><br>
                <span style="font-size:18px;">
                    x[n] = x[n+k]
                </span><br>
                porque k = mN para algún entero m.
            </div>
        `;
    }

    else if (isPeriodicSignal) {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #c44;
                background-color: #fff1f1;
            ">
                <b>Las señales no coinciden.</b><br>
                k = ${k} no es múltiplo de N = ${N}.<br><br>
                Para una señal periódica de período N:
                <br>
                <span style="font-size:18px;">
                    x[n] = x[n+k]
                </span>
                únicamente cuando k = mN.
            </div>
        `;
    }

    else {
        info.text = `
            <div style="
                padding: 10px;
                border-left: 5px solid #c44;
                background-color: #fff1f1;
            ">
                <b>Señal no periódica.</b><br>
                Aunque la parte oscilatoria utiliza N = ${N},
                la envolvente exponencial impide que la señal se repita.<br><br>
                En general:
                <span style="font-size:18px;">
                    x[n] ≠ x[n+k]
                </span>
                para k ≠ 0.
            </div>
        `;
    }

    source.change.emit();
    """
)

for widget, attribute in [
    (select, "value"),
    (amp_slider, "value"),
    (period_slider, "value"),
    (shift_slider, "value")
]:
    widget.js_on_change(attribute, callback)

# Ejecutamos el callback una vez para generar el texto inicial
initial_info = """
<div style="
    padding: 10px;
    border-left: 5px solid #2e8b57;
    background-color: #eef8f1;
">
    <b>Las señales coinciden.</b><br>
    k = 0 es múltiplo de N = 8.<br><br>
    <span style="font-size:18px;">x[n] = x[n+k]</span>
</div>
"""

info.text = initial_info

layout = column(
    p,
    row(select, amp_slider),
    row(period_slider, shift_slider),
    info
)

show(layout)